# Solutions · 10 · Adsorption and fixed beds

Worked solutions to the exercises of [notebook 10](../notebooks/10_adsorption_and_fixed_beds.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engsep is missing): install it from GitHub
    import engsep
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engsep"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engsep import adsorption as ad
from engsep import datasets
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

In [2]:
d = pd.read_csv(datasets.path("isotherm_phenol_carbon.csv"), comment="#")
lf = ad.fit_langmuir(d.c_mol_m3, d.q_mol_kg)
rho_b, eps, L, u, k = 500.0, 0.4, 1.0, 0.01, 6e-4

## Exercise 1

The phenol bed is fed at 5 mol/m3 instead of 10. Recompute the ideal breakthrough time and the Bohart-Adams curve (same k). Why does a lower feed concentration give a *longer* time but a *broader* curve, and how does the LUB change?

In [3]:
t = np.linspace(0, 40000, 800)
for c0 in (10.0, 5.0):
    q0 = float(ad.langmuir(c0, lf["q_max"], lf["K"]))
    t_star = ad.stoichiometric_time(q0, c0, rho_b, eps, L, u)
    curve = ad.bohart_adams(t, c0, q0, rho_b, L, u, k)
    an = ad.breakthrough_analysis(t, curve)
    print(f"c0 = {c0}: q0 = {q0:.2f} mol/kg, ideal breakthrough {t_star/3600:.2f} h; 5 % breakthrough {an['t_break']/3600:.2f} h, exhaustion {an['t_exhaust']/3600:.2f} h, LUB {an['LUB_fraction']*L:.2f} m")

c0 = 10.0: q0 = 1.88 mol/kg, ideal breakthrough 2.62 h; 5 % breakthrough 2.47 h, exhaustion 2.75 h, LUB 0.05 m
c0 = 5.0: q0 = 1.51 mol/kg, ideal breakthrough 4.20 h; 5 % breakthrough 3.92 h, exhaustion 4.46 h, LUB 0.07 m


Halving the feed concentration more than halves the loading rate (the isotherm is favourable, so the
loading falls less than the concentration), which lengthens the ideal breakthrough time; but the
mass-transfer zone spreads in proportion to $1/(kc_0)$, so the curve is broader and the unused bed longer.
The LUB depends on the feed as well as the packing - a caution when scaling from laboratory to plant at a
different concentration.

## Exercise 2

Design a two-bed system: one bed adsorbs while the other regenerates for 4 h. For the 10 mol/m3 feed at 1 cm/s, what bed length gives a breakthrough time of at least 4 h using the LUB from the laboratory column?

In [4]:
q0 = float(ad.langmuir(10.0, lf["q_max"], lf["K"]))
t = np.linspace(0, 40000, 800)
an = ad.breakthrough_analysis(t, ad.bohart_adams(t, 10.0, q0, rho_b, L, u, k))
LUB = an["LUB_fraction"] * L
from scipy import optimize
L_needed = optimize.brentq(lambda L_new: ad.lub_scale_up(LUB, L, an["t_break"], L_new)["t_break_new"] - 4 * 3600, L, 10.0)
print(f"LUB = {LUB:.3f} m; a breakthrough time of 4 h needs a bed of {L_needed:.2f} m ({ad.lub_scale_up(LUB, L, an['t_break'], L_needed)['used_fraction_new']:.0%} used at breakthrough)")

LUB = 0.052 m; a breakthrough time of 4 h needs a bed of 1.58 m (97% used at breakthrough)


The LUB method scales the laboratory column to the plant bed with one number; the 4 h bed is a little over
the proportional length because the unused part is fixed. Two such beds, one adsorbing while the other is
regenerated for 4 h, make a continuous unit.

## Exercise 3

**Implement it yourself:** compute the stoichiometric time of the laboratory breakthrough curve as the area above the curve (trapezoidal rule) and compare with `adsorption.breakthrough_analysis`.

In [5]:
b = pd.read_csv(datasets.path("breakthrough_curve.csv"), comment="#")
mask = np.ones(len(b), bool); mask[12] = False
t_, c_ = b.t_s[mask].to_numpy(), np.clip(b.c_over_c0[mask].to_numpy(), 0, 1)
area = np.trapezoid(1 - c_, t_) if hasattr(np, "trapezoid") else np.trapz(1 - c_, t_)
print(f"by hand: t* = {area/3600:.3f} h; library: {ad.breakthrough_analysis(t_, c_)['t_stoich']/3600:.3f} h")

by hand: t* = 2.792 h; library: 2.792 h


The stoichiometric time is the area above the breakthrough curve - the bed's total capacity divided by the
feed rate - and the trapezoidal rule on the cleaned data reproduces the library's value.